# Stage 5 — Ontology Routing

Routes each patient's **symptom tree** (Stage 3) branches to candidate diseases in a fixed ontology taxonomy (Infectious, Cardiovascular, Respiratory, Neurologic, GI, Renal, Constitutional), with a rough low/medium/high confidence per candidate.

**Input:** `symptom_tree_results.json` (Stage 3) | **Output:** `ontology_routing_results.json`


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    LLMNotAvailableError,
    check_llm,
    get_llm_config,
    load_symptom_tree_results,
    ontology_routing_agent,
    print_pipeline_banner,
    save_ontology_routing_results,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")

tree_results_df, patient_symptom_trees = load_symptom_tree_results()
print(f"Loaded symptom trees: {len(tree_results_df)} admissions")

## Route Symptom Trees to Ontology Candidates

In [ ]:
routed = []

for _, row in tree_results_df.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    tree = row["symptom_tree"]
    print(f"Ontology routing patient={patient_id} hadm_id={hadm_id}...")
    routing = ontology_routing_agent(
        symptom_tree=tree,
        admission_id=hadm_id,
        patient_id=patient_id,
        config=LLM_CONFIG,
    )
    routed.append({
        "patient_id": patient_id,
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "ontology_routing_method": routing.get("_method", "unknown"),
        "ontology_routing": routing,
        "n_candidates": len(routing.get("top_candidates", [])),
        "top_candidate": (routing.get("top_candidates") or [{}])[0].get("disease"),
    })

results_df = pd.DataFrame(routed)
results_df[["patient_id", "admission_id", "ontology_routing_method", "n_candidates", "top_candidate"]]

## Save Results

In [ ]:
out = save_ontology_routing_results(results_df)
print(f"Saved → {out}")